### 在get_wave_from_hdf5建立完資料後，每筆資料都會在輸出的資料夾中。
檔案名稱都會是trace_name，現在要將他們整合成張量，準備餵入模型。
預期張量形狀會是 [總事件數,所有測站量,3,3000]，且沒有資料的測站波型都要是零


In [2]:
"""
將每個子資料夾中的 .npy 波形檔案合併為一個 [166, 3, 3000] 的 tensor。
- 依照 all_station_info.json 中 key 的順序決定每個測站在 tensor 中的 index
- 從檔名中解析測站代碼（例如 1201040334_ALS_HL_SMT_10.npy → ALS）
- 找不到對應測站的檔案會跳過並印出警告
- 不足 166 個測站時補零 (zero-padding)
- 輸出格式為 .npy
"""

import os
import json
import numpy as np
from pathlib import Path

# ============ 設定區 ============
ROOT_DIR = Path(r"D:\filtered_by_Hualien_events_extracted_waveforms")
STATION_JSON = Path(r"C:\Users\user\Desktop\L_Earthquake_Model\earthquake_project\source\all_stastion_info.json")  # ← 修改為你的 JSON 路徑
OUTPUT_DIR = ROOT_DIR / "merged_tensors"
N_CHANNELS = 3
N_SAMPLES = 3000
# ================================


def load_station_order(json_path: Path) -> dict:
    """讀取 JSON，回傳 {測站代碼: index} 的對應表。"""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    station_to_idx = {name: i for i, name in enumerate(data.keys())}
    return station_to_idx


def parse_station_from_filename(filename: str) -> str:
    """
    從檔名解析測站代碼。
    例如：1201040334_ALS_HL_SMT_10.npy → ALS
          1201040334_CHN1_HL_SMT_10.npy → CHN1
          1201040334_ENAH_HL_BH_10.npy  → ENAH
    規則：取底線分割後的第二個欄位（index=1）
    """
    parts = filename.replace(".npy", "").split("_")
    return parts[1]


def merge_subfolder(subfolder_path: Path, station_to_idx: dict) -> tuple:
    """讀取子資料夾中所有 .npy 檔並依測站順序放入 tensor。"""
    n_stations = len(station_to_idx)
    tensor = np.zeros((n_stations, N_CHANNELS, N_SAMPLES), dtype=np.float32)

    npy_files = sorted(subfolder_path.glob("*.npy"))
    matched = 0
    skipped = []

    for npy_file in npy_files:
        station_code = parse_station_from_filename(npy_file.name)

        if station_code not in station_to_idx:
            skipped.append(f"{npy_file.name} (測站: {station_code})")
            continue

        data = np.load(npy_file)

        if data.shape != (N_CHANNELS, N_SAMPLES):
            print(f"  ⚠️  跳過 {npy_file.name}：維度為 {data.shape}，預期 ({N_CHANNELS}, {N_SAMPLES})")
            continue

        idx = station_to_idx[station_code]
        tensor[idx] = data
        matched += 1

    if skipped:
        print(f"  ⚠️  {len(skipped)} 個檔案找不到對應測站：")
        for s in skipped:
            print(f"      - {s}")

    return tensor, matched


def main():
    # 讀取測站順序
    station_to_idx = load_station_order(STATION_JSON)
    n_stations = len(station_to_idx)
    print(f"📋 測站清單：共 {n_stations} 個測站")
    print(f"📐 目標維度：[{n_stations}, {N_CHANNELS}, {N_SAMPLES}]")

    # 建立輸出資料夾
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # 找出所有子資料夾
    subfolders = sorted([d for d in ROOT_DIR.iterdir() if d.is_dir() and d != OUTPUT_DIR])

    if not subfolders:
        print("❌ 找不到任何子資料夾！")
        return

    print(f"📁 共找到 {len(subfolders)} 個子資料夾")
    print(f"📂 輸出目錄：{OUTPUT_DIR}")
    print("=" * 50)

    
    for subfolder in subfolders:
        npy_count = len(list(subfolder.glob("*.npy")))
        print(f"\n🔄 處理：{subfolder.name}（{npy_count} 個 .npy 檔案）")

        tensor, matched = merge_subfolder(subfolder, station_to_idx)

        # 儲存
        output_path = OUTPUT_DIR / f"{subfolder.name}.npy"
        np.save(output_path, tensor)

        fill_rate = matched / n_stations * 100
        print(f"  ✅ 已儲存：{output_path.name}  shape={tensor.shape}  填充率={fill_rate:.1f}% ({matched}/{n_stations})")
    
    print("\n" + "=" * 50)
    print("🎉 全部完成！")


if __name__ == "__main__":
    main()

📋 測站清單：共 166 個測站
📐 目標維度：[166, 3, 3000]
📁 共找到 1683 個子資料夾
📂 輸出目錄：D:\filtered_by_Hualien_events_extracted_waveforms\merged_tensors

🔄 處理：1201040334（25 個 .npy 檔案）
  ✅ 已儲存：1201040334.npy  shape=(166, 3, 3000)  填充率=15.1% (25/166)

🔄 處理：1201040653（51 個 .npy 檔案）
  ✅ 已儲存：1201040653.npy  shape=(166, 3, 3000)  填充率=30.7% (51/166)

🔄 處理：1201040656（18 個 .npy 檔案）
  ✅ 已儲存：1201040656.npy  shape=(166, 3, 3000)  填充率=10.8% (18/166)

🔄 處理：1201040659（66 個 .npy 檔案）
  ✅ 已儲存：1201040659.npy  shape=(166, 3, 3000)  填充率=39.8% (66/166)

🔄 處理：1201050148（12 個 .npy 檔案）
  ✅ 已儲存：1201050148.npy  shape=(166, 3, 3000)  填充率=7.2% (12/166)

🔄 處理：1201051545（35 個 .npy 檔案）
  ✅ 已儲存：1201051545.npy  shape=(166, 3, 3000)  填充率=21.1% (35/166)

🔄 處理：1201090425（58 個 .npy 檔案）
  ✅ 已儲存：1201090425.npy  shape=(166, 3, 3000)  填充率=34.9% (58/166)

🔄 處理：1201090616（26 個 .npy 檔案）
  ✅ 已儲存：1201090616.npy  shape=(166, 3, 3000)  填充率=15.7% (26/166)

🔄 處理：1201090706（43 個 .npy 檔案）
  ✅ 已儲存：1201090706.npy  shape=(166, 3, 3000)  填充率=25.9% (43/166)

🔄 處理：120

建立pga的tensor [事件總數,166(所有測站的pga)]

In [5]:
"""
建立 PGA 標籤 tensor，維度為 [事件總數, 166]。
- 遍歷每個子資料夾中的 .npy 檔案
- 用檔名（去掉 .npy）去 CSV 中比對 trace_name 欄位
- 取出該筆資料的 pga 值
- 依照 all_station_info.json 的 key 順序放到正確的 index
- 沒有資料的位置補零
- 輸出為 .npy
"""

import json
import numpy as np
import pandas as pd
from pathlib import Path

# ============ 設定區 ============
ROOT_DIR = Path(r"D:\filtered_by_Hualien_events_extracted_waveforms")

STATION_JSON = Path(r"C:\Users\user\Desktop\L_Earthquake_Model\earthquake_project\source\all_stastion_info.json")  # ← 修改為你的 JSON 路徑

CSV_PATH = Path(r"C:\Users\user\Desktop\L_Earthquake_Model\earthquake_project\source\filtered_by_Hualien_events.csv")
OUTPUT_PATH = ROOT_DIR / "pga_labels.npy"
# ================================


def load_station_order(json_path: Path) -> dict:
    """讀取 JSON，回傳 {測站代碼: index} 的對應表。"""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    station_to_idx = {name: i for i, name in enumerate(data.keys())}
    return station_to_idx


def parse_station_from_filename(filename: str) -> str:
    """從檔名解析測站代碼。例如：1201040334_ALS_HL_SMT_10.npy → ALS"""
    parts = filename.replace(".npy", "").split("_")
    return parts[1]


def main():
    # 1. 讀取測站順序
    station_to_idx = load_station_order(STATION_JSON)
    n_stations = len(station_to_idx)
    print(f"📋 測站清單：共 {n_stations} 個測站")

    # 2. 讀取 CSV
    df = pd.read_csv(CSV_PATH)

    # ⭐ 關鍵修正：去除欄位名稱的前後空格
    df.columns = df.columns.str.strip()

    # ⭐ 去除 trace_name 值的前後空格
    df["trace_name"] = df["trace_name"].astype(str).str.strip()

    print(f"📄 CSV 共 {len(df)} 筆資料")
    print(f"   清理後欄位：{list(df.columns[:5])} ... {list(df.columns[-3:])}")

    # 用 trace_name 建立快速查詢的 dict
    trace_to_pga = dict(zip(df["trace_name"], df["pga"]))
    print(f"   建立 trace_name → pga 查詢表：{len(trace_to_pga)} 筆")

    # 印出幾個範例確認
    sample_keys = list(trace_to_pga.keys())[:3]
    for k in sample_keys:
        print(f"   範例：trace_name='{k}' → pga={trace_to_pga[k]}")

    # 3. 找出所有子資料夾（每個子資料夾 = 一個事件）
    subfolders = sorted([
        d for d in ROOT_DIR.iterdir()
        if d.is_dir() and d.name != "merged_tensors"
    ])
    n_events = len(subfolders)
    print(f"\n📁 共找到 {n_events} 個事件（子資料夾）")
    print(f"📐 目標維度：[{n_events}, {n_stations}]")
    print("=" * 50)

    # 4. 建立 PGA tensor
    pga_tensor = np.zeros((n_events, n_stations), dtype=np.float32)

    total_matched = 0
    total_skipped_station = 0
    total_skipped_trace = 0

    for event_idx, subfolder in enumerate(subfolders):
        npy_files = sorted(subfolder.glob("*.npy"))
        matched = 0

        for npy_file in npy_files:
            # 解析測站代碼
            station_code = parse_station_from_filename(npy_file.name)

            if station_code not in station_to_idx:
                total_skipped_station += 1
                continue

            # 用檔名（去掉 .npy）去 CSV 查 pga
            trace_name = npy_file.stem  # e.g. "1201040334_ALS_HL_SMT_10"

            if trace_name not in trace_to_pga:
                total_skipped_trace += 1
                if event_idx < 3:  # 只印前幾個事件的警告
                    print(f"  ⚠️  CSV 找不到 trace_name: '{trace_name}'")
                continue

            pga_value = trace_to_pga[trace_name]
            idx = station_to_idx[station_code]
            pga_tensor[event_idx, idx] = pga_value
            matched += 1

        total_matched += matched

        if event_idx % 100 == 0 or event_idx == n_events - 1:
            fill_rate = matched / n_stations * 100
            print(f"  [{event_idx+1}/{n_events}] {subfolder.name}：{matched} 個測站有 PGA（填充率 {fill_rate:.1f}%）")

    # 5. 儲存
    np.save(OUTPUT_PATH, pga_tensor)

    print("\n" + "=" * 50)
    print(f"✅ PGA tensor 已儲存：{OUTPUT_PATH}")
    print(f"   shape = {pga_tensor.shape}")
    print(f"   總共匹配：{total_matched} 筆")
    print(f"   測站不在清單中跳過：{total_skipped_station} 筆")
    print(f"   CSV 找不到 trace_name：{total_skipped_trace} 筆")
    if pga_tensor[pga_tensor > 0].size > 0:
        print(f"   PGA 範圍：min={pga_tensor[pga_tensor > 0].min():.6f}, max={pga_tensor.max():.6f}")
    print("🎉 完成！")


if __name__ == "__main__":
    main()

📋 測站清單：共 166 個測站


C:\Users\user\AppData\Local\Temp\ipykernel_20616\4071236917.py:47: DtypeWarning: Columns (0:  trace_s_arrival_time) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


📄 CSV 共 129939 筆資料
   清理後欄位：['source_origin_time', 'source_depth_km', 'source_latitude_deg', 'source_longitude_deg', 'source_magnitude'] ... ['trace_completeness', 'year', 'pga']
   建立 trace_name → pga 查詢表：129916 筆
   範例：trace_name='1205202239_EGFH_HL_BH_10' → pga=1.429186476
   範例：trace_name='1205202239_ESL_HL_SMT_10' → pga=2.480711497
   範例：trace_name='1205202239_HWA_HL_BH_10' → pga=0.299020427

📁 共找到 1683 個事件（子資料夾）
📐 目標維度：[1683, 166]
  [1/1683] 1201040334：25 個測站有 PGA（填充率 15.1%）
  [101/1683] 1204040609：58 個測站有 PGA（填充率 34.9%）
  [201/1683] 1206181749：50 個測站有 PGA（填充率 30.1%）
  [301/1683] 1303120115：37 個測站有 PGA（填充率 22.3%）
  [401/1683] 1311011152：74 個測站有 PGA（填充率 44.6%）
  [501/1683] 1406251344：85 個測站有 PGA（填充率 51.2%）
  [601/1683] 1504220749：85 個測站有 PGA（填充率 51.2%）
  [701/1683] 1601010623：10 個測站有 PGA（填充率 6.0%）
  [801/1683] 1605021428：131 個測站有 PGA（填充率 78.9%）
  [901/1683] 1610212317：68 個測站有 PGA（填充率 41.0%）
  [1001/1683] 1802041410：136 個測站有 PGA（填充率 81.9%）
  [1101/1683] 1802061856：100 個測站有 PGA（填充率 

### 將wave與pga合併

In [ ]:
"""
將 merged_tensors/ 中的各事件 .npy 波形檔 + pga_labels.npy 合併為 HDF5。
輸出：
  - wave: [N, 166, 3, 3000]
  - pga:  [N, 166]
"""

import numpy as np
import h5py
from pathlib import Path

# ============ 設定區 ============
ROOT_DIR = Path(r"D:\filtered_by_Hualien_events_extracted_waveforms")
WAVE_DIR = ROOT_DIR / "merged_tensors"       # merge_waveforms.py 的輸出
PGA_PATH = ROOT_DIR / "pga_labels.npy"       # build_pga_labels.py 的輸出
OUTPUT_H5 = ROOT_DIR / "train_data.h5"
# ================================

def main():
    # 1. 讀取 PGA labels
    pga = np.load(PGA_PATH)
    print(f"📋 pga_labels shape: {pga.shape}")  # [N, 166]
    n_events = pga.shape[0]

    # 2. 找出所有波形 .npy（按檔名排序，與 pga_labels 的事件順序一致）
    wave_files = sorted(WAVE_DIR.glob("*.npy"))
    print(f"📁 找到 {len(wave_files)} 個波形檔案")

    if len(wave_files) != n_events:
        print(f"⚠️  警告：波形檔案數 ({len(wave_files)}) ≠ PGA 事件數 ({n_events})")
        print("   將以較小的數量為準")
        n_events = min(len(wave_files), n_events)
        wave_files = wave_files[:n_events]
        pga = pga[:n_events]

    # 3. 讀第一個檔案確認維度
    sample = np.load(wave_files[0])
    n_stations, n_channels, n_samples = sample.shape
    print(f"📐 單一事件 shape: {sample.shape}")
    print(f"📐 最終 wave shape: [{n_events}, {n_stations}, {n_channels}, {n_samples}]")
    print(f"📐 最終 pga  shape: [{n_events}, {n_stations}]")
    print("=" * 50)

    # 4. 寫入 HDF5
    with h5py.File(OUTPUT_H5, 'w') as hf:
        # 建立 dataset（一次分配好空間）
        wave_ds = hf.create_dataset(
            'wave',
            shape=(n_events, n_stations, n_channels, n_samples),
            dtype='float32',
            chunks=(1, n_stations, n_channels, n_samples),  # 逐筆讀取最快
            compression='gzip',
            compression_opts=4
        )
        pga_ds = hf.create_dataset(
            'pga',
            data=pga[:n_events].astype(np.float32),
            compression='gzip',
            compression_opts=4
        )

        # 逐筆寫入波形（避免一次載入全部到記憶體）
        for i, wf in enumerate(wave_files):
            wave_ds[i] = np.load(wf)
            if (i + 1) % 100 == 0 or i == n_events - 1:
                print(f"  寫入進度：[{i+1}/{n_events}]")

    file_size_mb = OUTPUT_H5.stat().st_size / (1024 * 1024)
    print("\n" + "=" * 50)
    print(f"✅ HDF5 已儲存：{OUTPUT_H5}")
    print(f"   檔案大小：{file_size_mb:.1f} MB")
    print(f"   wave: [{n_events}, {n_stations}, {n_channels}, {n_samples}]")
    print(f"   pga:  [{n_events}, {n_stations}]")
    print("🎉 完成！")


if __name__ == "__main__":
    main()